# Optional analysis and preprocessing steps of MePRAM dataset

## Process bacthecom data

In [ ]:
import msoffcrypto
import io
import pandas as pd

# Password to decrypt the password-protected BACTHECOM Excel file.
# Replace 'placeholder' with the actual password before running.
password = "placeholder"

# Open and decrypt the file using the msoffcrypto library
with open("/home/pmata/mepram_data/bacthecom_hc_urgencias_v2_completo.xlsx", "rb") as file:
    office_file = msoffcrypto.OfficeFile(file)
    office_file.load_key(password=password)
    decrypted = io.BytesIO()        # in-memory buffer to hold decrypted bytes
    office_file.decrypt(decrypted)

bacthecom_df = pd.read_excel(decrypted)

# Identify columns shared between BACTHECOM and the main resistance dataset (for alignment/validation)
shared_cols = [col for col in bacthecom_df.columns if col in df_resist.columns]
bact_df_shared = bacthecom_df.copy()[shared_cols]

### Load and filter BACTHECOM Excel file

Load the external BACTHECOM (Spanish bacteraemia cohort) Excel and retain only the columns shared with the main resistance dataset.

In [ ]:
# Exploratory: check the organism distribution in df_expanded after excluding rare categories
# (Enterococcus, fungi, and other rare bacteria are filtered out to focus on the main pathogens)
df_expanded[~df_expanded["resultado_hemo"].isin([
    "Enterococcus",
    "_Fungi",
    "_Other bacteria",
])]["resultado_hemo"].value_counts()

## Optional: Data analysis

### Plot results from hierarchical_model_train_rfecv.py execution in hpc

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import f1_score
# binary_optuna_stacking_cef_normal_20260127114340
results_root = Path(
    "/data/ucct/bi/scratch_tmp/bi/TESTS/pmata_mepram_tests/bmr_phenotest/outputs/binary_optuna_cef_new_20260202172017/"
)

subset_dirs = sorted(
    d for d in results_root.iterdir()
    if d.is_dir() and d.name.startswith("rfecv_")
)

def load_confusion_df(path: Path) -> pd.DataFrame | None:
    return pd.read_csv(path, index_col=0) if path.exists() else None


for subset_dir in subset_dirs:
    # --- detectar modo por nombre de archivo ---
    binary_cm = subset_dir / "binary_confusion_matrix.csv"
    multiclass_cm = subset_dir / "multiclass_confusion_matrix.csv"

    if binary_cm.exists():
        mode = "binary"
        cm_path = binary_cm
    if multiclass_cm.exists():
        mode = "multiclass"
        cm_path = multiclass_cm
    if binary_cm.exists() or multiclass_cm.exists():
        if binary_cm.exists() and multiclass_cm.exists():
            mode= "hierarchical"
            print("THIS IS A HIERARCHICAL TEST")
    else:
        print(f"No confusion matrix found in {subset_dir}")
        continue

    predictions_dirs = [f for f in subset_dir.iterdir() if "predictions.csv" in f.name]
    if len(predictions_dirs) == 1:
        predictions_df = pd.read_csv(predictions_dirs[0])
    elif len(predictions_dirs) > 1:
        raise ValueError("MULTIPLE PREDICTIONS CSV FOUND: ", str(predictions_dirs))
    else:
        predictions_df = None
        print("NO PREDICTION DIR FOUND")

    if predictions_df is not None:
        coldict = {
            "binary": {"true": "true_binary_label", "pred": "binary_pred"},
            "hierarchical": {"true": "true_fenotipo", "pred": "hierarchical_pred"}
        }
        if mode == "hierarchical":
            predictions_df = predictions_df[(predictions_df["true_fenotipo"] != "NEGATIVE") & (predictions_df["hierarchical_pred"] != "NEGATIVE")]
            print(predictions_df)
            f1_score_m = f1_score(
                predictions_df[coldict[mode]["true"]],
                predictions_df[coldict[mode]["pred"]],
                average="micro",
                sample_weight=predictions_df.get("sample_weight", 1),
            )
            print("f1_score: ", f1_score_m)

    df = load_confusion_df(cm_path)
    if df is None:
        print(f"Empty confusion matrix in {subset_dir}")
        continue

    # limpieza de etiquetas (solo relevante para multiclase)
    if mode == "multiclass":
        df = df.rename(
            columns=lambda x: x.split("resistente a ")[-1],
            index=lambda x: x.split("resistente a ")[-1],
        )

    # --- cargar summary ---
    summary_path = subset_dir / "summary.json"
    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
    summary["f1_score_micro"] = f1_score_m
    if mode == "binary":
        suffix = (
            f"\nBinary macro ROC-AUC: {summary.get('binary_roc_auc'):.3f}"
            if summary.get("binary_roc_auc") is not None
            else ""
        )
        cmap = "Blues"
        title = f"{subset_dir.name} – Binary classification{suffix}"

    else:
        suffix = (
            f"\nMulticlass macro ROC-AUC: {summary.get('multiclass_macro_auc'):.3f}"
            if summary.get("multiclass_macro_auc") is not None
            else ""
        )
        cmap = "Greens"
        title = f"{subset_dir.name} – Multiclass classification{suffix}"

    # --- plot ---
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.heatmap(df, annot=True, fmt=".2f", cmap=cmap, ax=ax)

    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")

    plt.tight_layout()
    plt.show()

    break  # quita esto si quieres todos los subsets
else:
    print("subset_dirs is empty")


In [ ]:
aucs, f1s = [], []
for root_dir in Path("/data/ucct/bi/scratch_tmp/bi/TESTS/pmata_mepram_tests/bmr_phenotest/outputs/").iterdir():

    if not "binary_optuna_cefalosporinas" in root_dir.name and not "other" in root_dir.name:
        continue
    subset_dirs = sorted(
        d for d in root_dir.iterdir()
        if d.is_dir() and d.name.startswith("rfecv_")
    )
    
    def load_confusion_df(path: Path) -> pd.DataFrame | None:
        return pd.read_csv(path, index_col=0) if path.exists() else None


    for subset_dir in subset_dirs:
        # --- detectar modo por nombre de archivo ---
        binary_cm = subset_dir / "binary_confusion_matrix.csv"
        multiclass_cm = subset_dir / "multiclass_confusion_matrix.csv"

        if binary_cm.exists():
            mode = "binary"
            cm_path = binary_cm
        elif multiclass_cm.exists():
            mode = "multiclass"
            cm_path = multiclass_cm
        else:
            print(f"No confusion matrix found in {subset_dir}")
            continue

        df = load_confusion_df(cm_path)
        if df is None:
            print(f"Empty confusion matrix in {subset_dir}")
            continue

        # limpieza de etiquetas (solo relevante para multiclase)
        if mode == "multiclass":
            df = df.rename(
                columns=lambda x: x.split("resistente a ")[-1],
                index=lambda x: x.split("resistente a ")[-1],
            )

        # --- cargar summary ---
        summary_path = subset_dir / "summary.json"
        summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
    print(root_dir.name)
    print(summary.get("binary_model"), summary.get("binary_roc_auc"))


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Point this to the run output root (e.g., outputs/binary_optuna_YYYYMMDDHHMMSS)
results_root = Path("/data/ucct/bi/scratch_tmp/bi/TESTS/pmata_mepram_tests/bmr_phenotest/outputs/binary_optuna_cefalosporinas_dropother_20251217120107/")
subset_dirs = sorted(
    d for d in results_root.iterdir()
    if d.is_dir() and d.name.startswith("rfecv_bin")
)

def load_confusion_df(path: Path) -> pd.DataFrame | None:
    return pd.read_csv(path, index_col=0) if path.exists() else None

print(subset_dirs)
for subset_dir in subset_dirs:
    preds_path = subset_dir / "binary_predictions.csv"
    if not preds_path.exists():
        print(f"Missing predictions: {preds_path}")
        continue

    preds = pd.read_csv(preds_path)
    classes = sorted(set(preds["true_binary_label"]) | set(preds["binary_pred"]))

    binary_df = load_confusion_df(subset_dir / "binary_confusion_matrix.csv")
    if binary_df is None:
        y_true = preds["true_binary_label"]
        y_pred = preds["binary_pred"]
        cm = confusion_matrix(y_true, y_pred, labels=classes)
        binary_df = pd.DataFrame(
            cm,
            index=[f"true_{lbl}" for lbl in classes],
            columns=[f"pred_{lbl}" for lbl in classes],
        )
    for col in binary_df:
        binary_df[col] = binary_df[col].astype(int)

    # Load summary for metrics
    summary_path = subset_dir / "summary.json"
    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
    auc_str = f"{summary.get('binary_roc_auc'):.3f}" if summary.get("binary_roc_auc") is not None else "N/A"
    f1_str = f"{summary.get('macro_f1'):.3f}" if summary.get("macro_f1") is not None else "N/A"

    fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    sns.heatmap(binary_df, annot=True, fmt=".0f", cmap="Blues", ax=ax)
    ax.set_title(f"{subset_dir.name}\nMacro F1: {f1_str} | ROC-AUC: {auc_str}")
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

    break  # remove this break to loop over all subsets
else:
    print("subset_dirs is empty")


In [ ]:
import ast
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
#hierarchical_optuna_bmr_multilabel_g2_20251216101932
results_root = Path("/data/ucct/bi/scratch_tmp/bi/TESTS/pmata_mepram_tests/bmr_phenotest/outputs/multilabel_optuna_bmr_stacking_fix_20260122085344/")
subset_dirs = sorted(
    d for d in results_root.iterdir()
    if d.is_dir() and d.name.startswith("rfecv_")
)

def load_confusion_df(path: Path) -> pd.DataFrame | None:
    return pd.read_csv(path, index_col=0) if path.exists() else None

def split_labels(series: pd.Series, delim: str = "|") -> list[list[str]]:
    def parse_entry(val: object) -> list[str]:
        if isinstance(val, list):
            return [str(x) for x in val if str(x)]
        as_str = "" if pd.isna(val) else str(val)
        stripped = as_str.strip()
        if stripped.startswith("[") and stripped.endswith("]"):
            try:
                parsed = ast.literal_eval(as_str)
                if isinstance(parsed, (list, tuple)):
                    return [str(x) for x in parsed if str(x)]
            except Exception:
                pass
        return [x for x in as_str.split(delim) if x]
    return series.fillna("").apply(parse_entry).tolist()

def ranked_labels_from_proba(series: pd.Series) -> list[list[str]]:
    ranked: list[list[str]] = []
    for val in series.fillna(""):
        labels: list[str] = []
        try:
            parsed = ast.literal_eval(val) if isinstance(val, str) else val
            multiclass = parsed.get("multiclass") if isinstance(parsed, dict) else {}
            if multiclass:
                labels = [lbl for lbl, _ in sorted(multiclass.items(), key=lambda x: x[1], reverse=True)]
        except Exception:
            labels = []
        ranked.append(labels)
    return ranked

def precision_recall_at_k(true_lists: list[list[str]], ranked_preds: list[list[str]], k: int) -> tuple[float, float]:
    precisions: list[float] = []
    recalls: list[float] = []
    for true_labels, pred_labels in zip(true_lists, ranked_preds):
        topk = pred_labels[:k]
        hit = len(set(true_labels) & set(topk))
        precisions.append(hit / len(topk) if topk else 0.0)
        recalls.append(hit / len(true_labels) if true_labels else 0.0)
    return (
        sum(precisions) / len(precisions) if precisions else float("nan"),
        sum(recalls) / len(recalls) if recalls else float("nan"),
    )

print(subset_dirs)
for subset_dir in subset_dirs:
    preds_path = subset_dir / "hierarchical_predictions.csv"
    if not preds_path.exists():
        print(f"Could not find {preds_path}")
        continue

    preds = pd.read_csv(preds_path)
    negative_label = next(iter(set(preds["binary_pred"]) - {"POSITIVE"}), "NEGATIVE")

    # Binary confusion
    binary_df = load_confusion_df(subset_dir / "binary_confusion_matrix.csv")
    if binary_df is None:
        y_true_bin = preds["true_binary_label"].ne(negative_label).astype(int)
        y_pred_bin = preds["binary_pred"].eq("POSITIVE").astype(int)
        cm = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1])
        binary_df = pd.DataFrame(
            cm,
            index=[f"true_{negative_label}", "true_POSITIVE"],
            columns=[f"pred_{negative_label}", "pred_POSITIVE"],
        )
    for col in binary_df:
        binary_df[col] = binary_df[col].astype(int)

    # Detect multi-label run
    multilabel_metrics_path = subset_dir / "multilabel_metrics.json"
    is_multilabel = multilabel_metrics_path.exists() or (preds["hierarchical_pred"].astype(str).str.contains(r"\|").any())

    summary_path = subset_dir / "summary.json"
    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
    binary_suffix = (
        f"\nBinary macro ROC-AUC: {summary.get('binary_roc_auc'):.3f}"
        if summary.get("binary_roc_auc") is not None
        else ""
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    sns.heatmap(binary_df, annot=True, fmt=".2f", cmap="Blues", ax=axes[0])
    axes[0].set_title(f"{subset_dir.name} – Binary{binary_suffix}")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("")

    if not is_multilabel:
        multiclass_df = load_confusion_df(subset_dir / "multiclass_confusion_matrix.csv")
        if multiclass_df is None:
            pos_mask = preds["true_binary_label"].ne(negative_label)
            multiclass_df = None
            if pos_mask.any():
                y_true_pos = preds.loc[pos_mask, "true_fenotipo"]
                y_pred_pos = preds.loc[pos_mask, "hierarchical_pred"]
                labels = sorted(set(y_true_pos) | set(y_pred_pos))
                cm = confusion_matrix(y_true_pos, y_pred_pos, labels=labels)
                multiclass_df = pd.DataFrame(
                    cm,
                    index=[f"true_{lbl}" for lbl in labels],
                    columns=[f"pred_{lbl}" for lbl in labels],
                )
        if multiclass_df is not None:
            sns.heatmap(multiclass_df, annot=True, fmt=".2f", cmap="Greens", ax=axes[1])
            axes[1].set_title(
                f"{subset_dir.name} – Multiclass "
                f"{'' if summary.get('multiclass_macro_auc') is None else f'(AUC {summary['multiclass_macro_auc']:.3f})'}"
            )
        else:
            axes[1].axis("off")
            axes[1].set_title(f"{subset_dir.name} – Multiclass")
    else:
        # Multi-label: show top label frequencies and metrics text
        pos_mask = preds["true_binary_label"].ne(negative_label)
        true_lists = split_labels(preds.loc[pos_mask, "true_fenotipo"])
        pred_lists = split_labels(preds.loc[pos_mask, "hierarchical_pred"])
        ranked_pred_lists = ranked_labels_from_proba(preds.loc[pos_mask, "hierarchical_proba"])
        true_flat = pd.Series([lbl for lst in true_lists for lbl in lst])
        pred_flat = pd.Series([lbl for lst in pred_lists for lbl in lst])
        top_true = true_flat.value_counts().head(10)
        top_pred = pred_flat.value_counts().head(10)
        freq_df = pd.DataFrame({"true": top_true, "pred": top_pred}).fillna(0).astype(int)
        freq_df.plot.bar(ax=axes[1], rot=90, title=f"{subset_dir.name} – Multi-label top-10 labels")
        axes[1].set_ylabel("count")

        ml_metrics = json.loads(multilabel_metrics_path.read_text()) if multilabel_metrics_path.exists() else {}

        precision_at_1_calc, recall_at_1_calc = precision_recall_at_k(true_lists, ranked_pred_lists, 1)
        precision_at_3_calc, recall_at_3_calc = precision_recall_at_k(true_lists, ranked_pred_lists, 3)

        precision_at_1_val = precision_at_1_calc if not pd.isna(precision_at_1_calc) else ml_metrics.get("precision_at_1", float("nan"))
        recall_at_1_val = recall_at_1_calc if not pd.isna(recall_at_1_calc) else ml_metrics.get("recall_at_1", float("nan"))
        precision_at_3_val = precision_at_3_calc if not pd.isna(precision_at_3_calc) else ml_metrics.get("precision_at_3", float("nan"))
        recall_at_3_val = recall_at_3_calc if not pd.isna(recall_at_3_calc) else ml_metrics.get("recall_at_3", float("nan"))

        text = "\n".join(
            [
                f"micro-F1: {ml_metrics.get('micro_f1', float('nan')):.3f}",
                f"macro-F1: {ml_metrics.get('macro_f1', float('nan')):.3f}",
                f"exact match: {ml_metrics.get('exact_match_ratio', float('nan')):.3f}",
                f"coverage: {ml_metrics.get('coverage', float('nan')):.3f}",
                f"inclusion_any_true: {ml_metrics.get('inclusion_any_true', float('nan')):.3f}",
                f"avg labels pred: {ml_metrics.get('avg_labels_predicted', float('nan')):.2f}",
                f"precision_at_top1: {precision_at_1_val:.2f}",
                f"recall_at_top1: {recall_at_1_val:.2f}",
                f"precision_at_top3: {precision_at_3_val:.2f}",
                f"recall_at_top3: {recall_at_3_val:.2f}",
            ]
        )
        axes[1].text(1.05, 0.5, text, transform=axes[1].transAxes, va="center")

    plt.tight_layout()
    plt.show()
    break
else:
    print("subset_dirs is empty")


In [ ]:
summary

In [ ]:
import json
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

sns.set_theme(style="whitegrid")

results_root = Path("/data/ucct/bi/scratch_tmp/bi/TESTS/pmata_mepram_tests/bmr_phenotest/outputs/hierarchical_optuna_bmr_multilabel_20251126113557/")
subset_dirs = sorted(d for d in results_root.iterdir() if d.is_dir() and d.name.startswith("rfecv_"))
if not subset_dirs:
    raise SystemExit("No subset dirs found")
subset_dir = subset_dirs[0]  # pick the first; change if needed

preds = pd.read_csv(subset_dir / "hierarchical_predictions.csv")
negative_label = next(iter(set(preds["binary_pred"]) - {"POSITIVE"}), "NEGATIVE")

# --- Binary confusion
binary_df = pd.read_csv(subset_dir / "binary_confusion_matrix.csv", index_col=0) if (subset_dir / "binary_confusion_matrix.csv").exists() else None
if binary_df is None:
    y_true_bin = preds["true_binary_label"].ne(negative_label).astype(int)
    y_pred_bin = preds["binary_pred"].eq("POSITIVE").astype(int)
    cm = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1])
    binary_df = pd.DataFrame(cm,
        index=[f"true_{negative_label}", "true_POSITIVE"],
        columns=[f"pred_{negative_label}", "pred_POSITIVE"],
    )
binary_df = binary_df.astype(int)

# --- Multi-label prep (positive samples only)
pos_mask = preds["true_binary_label"].ne(negative_label)
true_lists = preds.loc[pos_mask, "true_fenotipo"].fillna("").apply(lambda s: [x for x in str(s).split("|") if x]).tolist()
pred_lists = preds.loc[pos_mask, "hierarchical_pred"].fillna("").apply(lambda s: [x for x in str(s).split("|") if x]).tolist()

# Per-label recall
true_counts = Counter(lbl for lst in true_lists for lbl in lst)
correct_counts = Counter()
for tl, pl in zip(true_lists, pred_lists):
    ps = set(pl)
    for lbl in tl:
        if lbl in ps:
            correct_counts[lbl] += 1
recall_df = pd.DataFrame([
    {"label": lbl, "recall": correct_counts.get(lbl, 0) / cnt, "support": cnt}
    for lbl, cnt in true_counts.items()
]).sort_values("recall", ascending=False)

# Top labels (true vs pred)
true_flat = Counter(lbl for lst in true_lists for lbl in lst)
pred_flat = Counter(lbl for lst in pred_lists for lbl in lst)
top_true = true_flat.most_common(7)
top_pred = pred_flat.most_common(7)
top_labels = list({lbl for lbl, _ in top_true} | {lbl for lbl, _ in top_pred})
top_df = pd.DataFrame({
    "true": {lbl: true_flat.get(lbl, 0) for lbl in top_labels},
    "pred": {lbl: pred_flat.get(lbl, 0) for lbl in top_labels},
}).reindex(index=sorted(top_labels))

# Label count per patient + coverage/hit-any
true_counts_per = [len(x) for x in true_lists]
pred_counts_per = [len(x) for x in pred_lists]
coverage = sum(c > 0 for c in pred_counts_per) / len(pred_counts_per) if pred_counts_per else 0.0
hit_any = sum(len(set(t) & set(p)) > 0 for t, p in zip(true_lists, pred_lists)) / len(true_lists) if true_lists else 0.0

# Optional saved metrics
ml_metrics = {}
ml_path = subset_dir / "multilabel_metrics.json"
if ml_path.exists():
    ml_metrics = json.loads(ml_path.read_text())

# --- Plotting (4 panels)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.heatmap(binary_df, annot=True, fmt="d", cmap="Blues", ax=axes[0, 0])
axes[0, 0].set_title(f"{subset_dir.name} – Binary gate")
axes[0, 0].set_xlabel("")
axes[0, 0].set_ylabel("")

sns.barplot(
    data=recall_df,
    x="recall",
    y="label",
    order=recall_df["label"].tolist(),
    palette="Greens_r",
    ax=axes[0, 1],
)
axes[0, 1].set_title("Per-label recall (multi-label)")
axes[0, 1].set_xlim(0, 1)

top_df.plot.bar(ax=axes[1, 0], rot=45, title="Top labels: true vs predicted")
axes[1, 0].set_ylabel("count")

axes[1, 1].hist(true_counts_per, bins=range(0, max(true_counts_per + pred_counts_per + [1]) + 1), alpha=0.6, label="true")
axes[1, 1].hist(pred_counts_per, bins=range(0, max(true_counts_per + pred_counts_per + [1]) + 1), alpha=0.6, label="pred")
axes[1, 1].set_title(f"Labels per patient\ncoverage(any pred): {coverage:.2f} | hit(any correct): {hit_any:.2f}")
axes[1, 1].set_xlabel("# labels")
axes[1, 1].set_ylabel("count")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

if ml_metrics:
    print("Saved multi-label metrics:")
    for k, v in ml_metrics.items():
        print(f"  {k}: {v}")


In [ ]:
foco_map = {1.0: 'pulmonar',
 2.0: 'intraabdominal',
 3.0: 'biliar',
 4.0: 'urinario',
 5.0: 'cardiovascular',
 6.0: 'piel',
 7.0: 'sistema nervioso central',
 8.0: 'catéter venoso',
 9.0: 'vías altas respiratorias',
 10.0: 'osteoarticular',
 11.0: 'genital',
 12.0: 'desconocido'}

## Visualisation setup

Load plotting libraries and helper metadata that support the upcoming exploratory analyses.

In [ ]:
sintom_dict = tbl_codes2names[tbl_codes2names["name"].str.contains("síntoma ")][["value", "name"]].to_dict(orient="records")
sintom_dict = {"sintoma_"+str(float(d["value"])): "sintoma_" + d["name"].replace("síntoma | ", "") for d in sintom_dict}
sintom_map = {}
for k,v in sintom_dict.items():
    sintom_map[k] = v
    sintom_map[k+"_categorico"] = v+"_categorico"
sintom_map

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import LabelEncoder
import copy

def correlation_ratio(categories, measurements):
    """Correlation ratio (eta squared) between categorical target and numerical feature"""
    categories = np.array(categories)
    measurements = np.array(measurements)
    cat_groups = [measurements[categories == cat] for cat in np.unique(categories)]
    means = [np.mean(g) for g in cat_groups if len(g) > 0]
    n = len(measurements)
    grand_mean = np.mean(measurements)
    ss_between = sum(len(g) * (m - grand_mean) ** 2 for g, m in zip(cat_groups, means))
    ss_total = sum((measurements - grand_mean) ** 2)
    return np.sqrt(ss_between / ss_total) if ss_total > 0 else 0

# -------------------------------------
target = "resultado_hemo"
df_to_plot = df_expanded.copy().rename(columns=sintom_map)
df = copy.deepcopy(df_to_plot.drop(["freq_bac_foco", "freq_bacteria", "infected_yes_no", "person_id"], axis=1))
df = df[~df["resultado_hemo"].isin(["Enterococcus", "_Fungi", "_Other bacteria", '_Virus'])]

antibmap = {x["value"]:x["name"].split("|")[1].strip() for x in tbl_codes2names[tbl_codes2names["variable"] == "antimicrobiano_previo"][["value", "name"]].to_dict(orient="records")}
df["ultimo_antib"] = df["ultimo_antib"].map(antibmap)
df["ultimo_antib"] = LabelEncoder().fit_transform(df["ultimo_antib"])
#df = pd.get_dummies(df, columns=["ultimo_antib"], drop_first=True)
# Get categories directly
classes = df[target].unique()
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != target]
"""df = pd.get_dummies(df, columns=["ultimo_antib"], drop_first=True)
df = df.rename(columns=lambda c: c.replace('ultimo_antib_', ''))
numeric_cols = df.drop(target, axis=1).columns"""
# Prepare a dataframe to store correlations per class and feature
corr_matrix = pd.DataFrame(index=classes, columns=numeric_cols, dtype=float)

for cls in classes:
    # binary target: this class vs the rest
    y_binary = (df[target] == cls).astype(int)
    for col in numeric_cols:
        corr_matrix.loc[cls, col] = correlation_ratio(y_binary, df[col].values)

# Optional: filter weak correlations
filtered = corr_matrix.loc[:, corr_matrix.max(axis=0) > 0.1]

# -------------------------------------
# 🔹 Plot heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(filtered, annot=False, cmap="viridis", cbar_kws={"label": "η (class-wise correlation)"})
plt.title(f"Ultimo_antib correlation ratio (η) by etiology. Showing only those with η > 0.02")
plt.xlabel("Feature")
plt.ylabel("Bacteria / Etiology class")
plt.tight_layout()
plt.show()


## Subset for etiology modelling

Create a reduced dataframe focused on pre-admission events (infections, colonisations) paired with hemoculture outcomes.

In [ ]:
df_hemo_merged = pd.merge(df_pacientes, hemo_urg_pivoted[["person_id", "resultado_hemo"]], on= ['person_id'], how= 'left')

orig_cols = df_hemo_merged.columns.tolist()
df_hemo_merged = df_hemo_merged.merge(tbl_infecciones_complete, on= ['person_id'], how= 'left')
df_hemo_merged = df_hemo_merged.merge(colo_prev_pivoted, on = ['person_id'], how= 'left')
new_cols = [c for c in df_hemo_merged.columns if c not in orig_cols]
df_hemo_merged[new_cols] = df_hemo_merged[new_cols].fillna(0) # Not all patients have previous infections/colonizations

df_hemo_merged = df_hemo_merged.merge(tbl_sepsis, on= ['person_id', 'fecha_ingreso_urgencias'], how= 'left')

In [ ]:
df_expanded = df_hemo_merged.explode("resultado_hemo").reset_index(drop=True)
counts = df_expanded["person_id"].value_counts()
df_expanded["weight"] = df_expanded["person_id"].apply(lambda x: 1 / counts[x])
df_expanded["co_infection"] = np.where(df_expanded["weight"] < 1, 1, 0)

In [ ]:
df_expanded.to_csv(os.path.join(save_location, "df_hemo_merged_v2.csv"), index=False)

## Generate automated EDA report

Reload the merged dataset and render an HTML profiling report with `ydata_profiling`.

In [ ]:
df_merged = pd.read_csv(os.path.join(save_location, "df_merged_full.csv"))

In [ ]:
profile = ProfileReport(df_merged, title="MePRAM EDA report")
profile.to_notebook_iframe()
profile.to_file(os.path.join(save_location, "df_merge_report.html"))

## Visualise missingness patterns

Plot heatmaps that highlight columns with substantial fractions of missing data to guide imputation strategies.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def insert_linebreak(string, lengLabel=10):
    return '\n'.join(string[i:i+lengLabel] for i in range(0, len(string), lengLabel))
merged_df = pd.read_csv(os.path.join(save_location, "df_merged.csv"))
prev_num = 0

def plot_missing_columns(subset_df, title=f"Missing Data Matrix for columns 0 to 171"):
    missing_df = subset_df.isna()

    missing_df.columns = subset_df.columns
    missing_percent = missing_df.mean() * 100
    columns_with_percent = [
        f"$\\bf{{{col}}}$ ({missing_percent[col]:.2f}%)" if missing_percent[col] > 30 else f"{col} ({missing_percent[col]:.2f}%)"
        for col in subset_df.columns
    ]
    plt.figure(figsize=(24, 6))
    ax = sns.heatmap(missing_df, vmin=0, vmax=1, cbar=False,
                xticklabels=columns_with_percent)
    ax.tick_params(axis='x', which='minor', length=40)
    plt.title(title)
    plt.xlabel("Columns (% Missing)")
    plt.ylabel("Rows", rotation=90)
    plt.xticks(rotation=90)
    plt.show()


plot_missing_columns(merged_df)

missing_df = merged_df.isna()
missing_df.columns = merged_df.columns
missing_percent = missing_df.mean() * 100
print([idx for idx,x in enumerate(missing_percent) if x > 30])
dangerous_df = merged_df.iloc[:, [idx for idx,x in enumerate(missing_percent) if x > 30]]
print(dangerous_df)
plot_missing_columns(dangerous_df, f"Missing Data Matrix of {len(dangerous_df.columns)} columns with > 30% NAs")

## Principal component analysis for feature exploration

Scale features with `MinMaxScaler`, fit 2D/3D PCA components, and visualise them interactively.

In [ ]:
TARGET_VARIABLE = "sepsis"

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

target = df_merged[TARGET_VARIABLE]
data = df_merged.drop(columns= [TARGET_VARIABLE])

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

pca = PCA(n_components=2)
pca_data = pca.fit_transform(scaled_data)

explained_variance = pca.explained_variance_ratio_
print(f"Varianza explicada por cada componente: {explained_variance}")
print(f"Varianza total explicada: {sum(explained_variance)}")

plt.figure(figsize=(8, 6))
plt.scatter(pca_data[:, 0], pca_data[:, 1], c=target, cmap='viridis', alpha=0.7)
plt.title("PCA 2D")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid()
plt.show()


In [ ]:
import plotly.express as px 

pca_3d = PCA(n_components=3)
pca_data_3d = pca_3d.fit_transform(scaled_data)

pca_df = pd.DataFrame(pca_data_3d, columns=["PC1", "PC2" ,"PC3"])
pca_df[TARGET_VARIABLE] = target

fig = px.scatter_3d(
    pca_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color = TARGET_VARIABLE,
    title = "PCA - 3D Visualization",
    labels= {TARGET_VARIABLE},
    color_continuous_scale="Viridis", 
    opacity=0.7  
)
fig.update_traces(marker=dict(size=5))  
fig.update_layout(scene=dict(
    xaxis_title="PC 1",
    yaxis_title="PC 2",
    zaxis_title="PC 3"
))
fig.show()

In [ ]:
processed_df_copy = df_merged

target_copy = processed_df_copy[TARGET_VARIABLE]
scaler = MinMaxScaler()
X_preprocessed = pd.DataFrame(scaler.fit_transform(processed_df_copy.drop(columns=[TARGET_VARIABLE])))
X_preprocessed[TARGET_VARIABLE] = target_copy

target_palette = {0: "blue", 1: "red"}
row_colors = X_preprocessed[TARGET_VARIABLE].map(target_palette)
X_preprocessed = X_preprocessed.dropna() 
sns.clustermap(
    X_preprocessed.drop(columns=[TARGET_VARIABLE]), 
    cmap="coolwarm",
    row_colors=row_colors,
    figsize=(30, 60),
    annot=False,
    col_cluster=False
)

plt.title("Heatmap con Clustering y Anotación por Target", pad=100)
plt.show()